# Wear Detection Exploratory Analysis

This notebook walks through the wear detection pipeline step-by-step, allowing interactive exploration and visualization at each stage.

## Pipeline Overview

1. **Load Data** - Read accelerometer, GPS, and gyroscope files
2. **Estimate Noise** - Determine sensor noise σ from stationary periods
3. **Detect Wear** - Apply hypothesis test to classify each minute
4. **Summarize** - Compute daily totals and hourly patterns
5. **Validate** - Check concordance with GPS and gyroscope

## Setup

In [ ]:
# Standard imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from datetime import datetime, timedelta
import pytz
from scipy import stats

# Add src to path
import sys
sys.path.insert(0, '../src')

# Import our modules
from data_loader import (
    load_subject_data, 
    get_subject_ids,
    compute_magnitude,
    convert_timezone
)
from wear_detection import (
    WearDetectionParams,
    estimate_noise_sigma_from_stationary,
    detect_wear_minutes,
    compute_daily_wear_summary,
    compute_hourly_wear_pattern,
    get_critical_value
)
from concordance import (
    compute_gps_displacement,
    compute_gyro_activity,
    merge_sensor_data,
    compute_concordance_metrics
)

# Plotting settings
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 11

print("Setup complete!")

## Configuration

Set your data paths and analysis parameters here.

In [ ]:
# ============================================================
# CONFIGURE THESE PATHS FOR YOUR DATA
# ============================================================

# Path to your data directory (containing subject folders)
DATA_DIR = Path('/path/to/your/data')  # <-- CHANGE THIS

# Subject to analyze (will list available subjects below)
SUBJECT_ID = 'subject_001'  # <-- CHANGE THIS

# Participant's timezone
TIMEZONE = 'US/Eastern'

# Hours assumed to be stationary for noise estimation (24h format)
STATIONARY_HOURS = (2, 5)  # 2 AM to 5 AM

# Significance level for hypothesis test
ALPHA = 0.05

# Minimum samples per minute to consider valid
MIN_SAMPLES_PER_MINUTE = 100  # At 10 Hz, expect ~600 samples/min

In [ ]:
# List available subjects
try:
    subjects = get_subject_ids(DATA_DIR)
    print(f"Found {len(subjects)} subjects:")
    for s in subjects:
        print(f"  - {s}")
except FileNotFoundError as e:
    print(f"ERROR: {e}")
    print("Please update DATA_DIR in the cell above.")

---

## Step 1: Load Accelerometer Data

In [ ]:
# Load accelerometer data for the subject
print(f"Loading accelerometer data for {SUBJECT_ID}...")
acc_df = load_subject_data(DATA_DIR, SUBJECT_ID, 'accelerometer')

print(f"\nLoaded {len(acc_df):,} samples")
print(f"Date range: {acc_df['timestamp'].min()} to {acc_df['timestamp'].max()}")
print(f"\nFirst few rows:")
acc_df.head()

In [ ]:
# Compute magnitude
acc_df = compute_magnitude(acc_df)

# Convert to local timezone for analysis
acc_local = convert_timezone(acc_df, TIMEZONE)

print(f"Magnitude range: {acc_df['magnitude'].min():.3f} to {acc_df['magnitude'].max():.3f} g")
print(f"Mean magnitude: {acc_df['magnitude'].mean():.3f} g")

### Visualize Raw Accelerometer Data

In [ ]:
# Plot a sample of raw data (first hour)
sample = acc_df.head(36000)  # ~1 hour at 10 Hz

fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=True)

# Plot x, y, z
ax1 = axes[0]
ax1.plot(sample['timestamp'], sample['x'], label='X', alpha=0.7, linewidth=0.5)
ax1.plot(sample['timestamp'], sample['y'], label='Y', alpha=0.7, linewidth=0.5)
ax1.plot(sample['timestamp'], sample['z'], label='Z', alpha=0.7, linewidth=0.5)
ax1.set_ylabel('Acceleration (g)')
ax1.set_title('Raw Accelerometer Data (X, Y, Z)')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Plot magnitude
ax2 = axes[1]
ax2.plot(sample['timestamp'], sample['magnitude'], color='purple', alpha=0.7, linewidth=0.5)
ax2.axhline(y=1.0, color='red', linestyle='--', alpha=0.5, label='1g (stationary)')
ax2.set_ylabel('Magnitude (g)')
ax2.set_xlabel('Time')
ax2.set_title('Acceleration Magnitude')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Distribution of magnitude values
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogram
ax1 = axes[0]
ax1.hist(acc_df['magnitude'], bins=100, edgecolor='black', alpha=0.7)
ax1.axvline(x=1.0, color='red', linestyle='--', label='1g')
ax1.set_xlabel('Magnitude (g)')
ax1.set_ylabel('Frequency')
ax1.set_title('Distribution of Acceleration Magnitude')
ax1.legend()

# Zoomed histogram around 1g
ax2 = axes[1]
near_1g = acc_df['magnitude'][(acc_df['magnitude'] > 0.9) & (acc_df['magnitude'] < 1.1)]
ax2.hist(near_1g, bins=50, edgecolor='black', alpha=0.7)
ax2.axvline(x=1.0, color='red', linestyle='--', label='1g')
ax2.set_xlabel('Magnitude (g)')
ax2.set_ylabel('Frequency')
ax2.set_title('Zoomed: Magnitude Near 1g (Stationary)')
ax2.legend()

plt.tight_layout()
plt.show()

---

## Step 2: Estimate Noise Parameter (σ)

We estimate the sensor noise from periods when the device is likely stationary (e.g., overnight).

In [ ]:
# Estimate noise from stationary periods
try:
    noise_sigma = estimate_noise_sigma_from_stationary(
        acc_local,
        stationary_hours=STATIONARY_HOURS,
        method='pooled_std'
    )
    print(f"Estimated noise σ: {noise_sigma:.6f} g ({noise_sigma * 1000:.3f} mg)")
except ValueError as e:
    print(f"Could not estimate noise: {e}")
    print("Using fallback value from literature (~10 mg for smartphone accelerometers)")
    noise_sigma = 0.01

In [ ]:
# Visualize the stationary period data used for noise estimation
start_hour, end_hour = STATIONARY_HOURS
stationary_mask = (
    (acc_local['timestamp'].dt.hour >= start_hour) & 
    (acc_local['timestamp'].dt.hour < end_hour)
)
stationary_data = acc_local[stationary_mask]

print(f"Stationary period data: {len(stationary_data):,} samples")

if len(stationary_data) > 0:
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    
    for i, col in enumerate(['x', 'y', 'z']):
        ax = axes[i]
        data = stationary_data[col]
        ax.hist(data, bins=50, edgecolor='black', alpha=0.7, density=True)
        ax.axvline(data.mean(), color='red', linestyle='--', 
                   label=f'Mean: {data.mean():.4f}')
        ax.set_xlabel(f'{col.upper()} acceleration (g)')
        ax.set_ylabel('Density')
        ax.set_title(f'{col.upper()}-axis during stationary period\nσ = {data.std():.4f} g')
        ax.legend()
    
    plt.tight_layout()
    plt.show()
else:
    print("No data available during stationary hours.")

---

## Step 3: Understand the Hypothesis Test

Before running the wear detection, let's understand the hypothesis test framework.

In [ ]:
# Critical value for chi-squared(3) at our alpha level
critical_value = get_critical_value(ALPHA, df=3)

print(f"Hypothesis Test Setup:")
print(f"  H₀: Device is stationary (M²/σ² ~ χ²₃)")
print(f"  H₁: Device is moving")
print(f"")
print(f"Parameters:")
print(f"  Noise σ = {noise_sigma:.6f} g")
print(f"  α = {ALPHA}")
print(f"  Critical value (χ²₃, 1-α) = {critical_value:.3f}")
print(f"")
print(f"Decision Rule:")
print(f"  If T = M²/σ² > {critical_value:.3f}, reject H₀ → classify as WEAR")
print(f"  If T = M²/σ² ≤ {critical_value:.3f}, fail to reject H₀ → classify as NON-WEAR")

In [ ]:
# Visualize the chi-squared distribution and critical region
x = np.linspace(0, 20, 1000)
y = stats.chi2.pdf(x, df=3)

fig, ax = plt.subplots(figsize=(10, 5))

ax.plot(x, y, 'b-', linewidth=2, label='χ²(3) distribution')
ax.fill_between(x[x > critical_value], y[x > critical_value], 
                alpha=0.3, color='red', label=f'Rejection region (α={ALPHA})')
ax.axvline(critical_value, color='red', linestyle='--', 
           label=f'Critical value = {critical_value:.2f}')

# Show what magnitude corresponds to the critical value
critical_magnitude = np.sqrt(critical_value) * noise_sigma
ax.set_xlabel('Test Statistic T = M²/σ²')
ax.set_ylabel('Probability Density')
ax.set_title(f'Hypothesis Test: χ²(3) Distribution\n'
             f'Critical magnitude ≈ {critical_magnitude:.4f} g')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

---

## Step 4: Run Wear Detection

In [ ]:
# Create parameters
params = WearDetectionParams(
    noise_sigma=noise_sigma,
    alpha=ALPHA,
    aggregation_method='mean',
    min_samples_per_minute=MIN_SAMPLES_PER_MINUTE
)

# Run wear detection
print("Running wear detection...")
wear_df = detect_wear_minutes(acc_df, params)

print(f"\nClassified {len(wear_df)} minutes")
print(f"\nSummary:")
print(f"  Valid minutes: {wear_df['valid'].sum()}")
print(f"  Invalid minutes: {(~wear_df['valid']).sum()}")
print(f"  Wear minutes: {wear_df['wear'].sum()}")
print(f"  Non-wear minutes: {(wear_df['valid'] & ~wear_df['wear']).sum()}")

In [ ]:
# Examine the wear detection results
print("Sample of wear detection results:")
wear_df.head(20)

In [ ]:
# Distribution of test statistics
valid_wear = wear_df[wear_df['valid']]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogram of test statistics
ax1 = axes[0]
ax1.hist(valid_wear['test_statistic'], bins=50, edgecolor='black', alpha=0.7)
ax1.axvline(critical_value, color='red', linestyle='--', 
            label=f'Critical value = {critical_value:.2f}')
ax1.set_xlabel('Test Statistic')
ax1.set_ylabel('Frequency')
ax1.set_title('Distribution of Test Statistics')
ax1.legend()
ax1.set_xlim(0, min(valid_wear['test_statistic'].quantile(0.99), 1000))

# Histogram of p-values
ax2 = axes[1]
ax2.hist(valid_wear['p_value'], bins=50, edgecolor='black', alpha=0.7)
ax2.axvline(ALPHA, color='red', linestyle='--', label=f'α = {ALPHA}')
ax2.set_xlabel('P-value')
ax2.set_ylabel('Frequency')
ax2.set_title('Distribution of P-values')
ax2.legend()

plt.tight_layout()
plt.show()

In [ ]:
# Visualize wear/non-wear over time
fig, ax = plt.subplots(figsize=(14, 4))

# Convert to local time for plotting
wear_local = wear_df.copy()
wear_local['local_time'] = wear_local['minute_start'].dt.tz_convert(TIMEZONE)

# Plot wear status
colors = wear_local['wear'].map({True: 'green', False: 'red', None: 'gray'})
ax.scatter(wear_local['local_time'], 
           wear_local['mean_magnitude'], 
           c=colors, alpha=0.5, s=5)

ax.axhline(y=1.0, color='blue', linestyle='--', alpha=0.5, label='1g')
ax.set_xlabel('Time (local)')
ax.set_ylabel('Mean Magnitude (g)')
ax.set_title('Wear Classification Over Time\n(Green=Wear, Red=Non-wear, Gray=Invalid)')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

---

## Step 5: Daily and Hourly Summaries

In [ ]:
# Compute daily summary
daily_summary = compute_daily_wear_summary(wear_df, timezone=TIMEZONE)

print("Daily Wear Summary:")
daily_summary

In [ ]:
# Visualize daily wear patterns
if len(daily_summary) > 0:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # Daily wear minutes
    ax1 = axes[0]
    x = range(len(daily_summary))
    ax1.bar(x, daily_summary['wear_minutes'], label='Wear', color='green', alpha=0.7)
    ax1.bar(x, daily_summary['nonwear_minutes'], bottom=daily_summary['wear_minutes'],
            label='Non-wear', color='red', alpha=0.7)
    ax1.set_xticks(x)
    ax1.set_xticklabels(daily_summary['date'], rotation=45, ha='right')
    ax1.set_ylabel('Minutes')
    ax1.set_title('Daily Wear Time')
    ax1.legend()
    
    # Daily wear fraction
    ax2 = axes[1]
    ax2.bar(x, daily_summary['wear_fraction'] * 100, color='blue', alpha=0.7)
    ax2.set_xticks(x)
    ax2.set_xticklabels(daily_summary['date'], rotation=45, ha='right')
    ax2.set_ylabel('Wear Fraction (%)')
    ax2.set_title('Daily Wear Percentage')
    ax2.set_ylim(0, 100)
    ax2.axhline(y=50, color='red', linestyle='--', alpha=0.5)
    
    plt.tight_layout()
    plt.show()

In [ ]:
# Compute hourly pattern
hourly_pattern = compute_hourly_wear_pattern(wear_df, timezone=TIMEZONE)

print("Hourly Wear Pattern:")
hourly_pattern

In [ ]:
# Visualize hourly wear pattern
fig, ax = plt.subplots(figsize=(12, 5))

ax.bar(hourly_pattern['hour'], hourly_pattern['wear_rate'] * 100, 
       color='steelblue', alpha=0.7, edgecolor='black')

ax.set_xlabel('Hour of Day (local time)')
ax.set_ylabel('Wear Rate (%)')
ax.set_title(f'Hourly Wear Pattern ({TIMEZONE})')
ax.set_xticks(range(24))
ax.set_xlim(-0.5, 23.5)
ax.set_ylim(0, 100)
ax.grid(True, alpha=0.3, axis='y')

# Highlight nighttime
ax.axvspan(-0.5, 6, alpha=0.1, color='gray', label='Night (0-6)')
ax.axvspan(22, 23.5, alpha=0.1, color='gray')

plt.tight_layout()
plt.show()

---

## Step 6: Cross-Sensor Concordance

Compare accelerometer wear classifications with GPS and gyroscope data.

In [ ]:
# Load GPS data
try:
    gps_df = load_subject_data(DATA_DIR, SUBJECT_ID, 'gps')
    print(f"Loaded {len(gps_df)} GPS samples")
    print(f"GPS date range: {gps_df['timestamp'].min()} to {gps_df['timestamp'].max()}")
    gps_activity = compute_gps_displacement(gps_df)
except FileNotFoundError:
    print("No GPS data found for this subject")
    gps_df = pd.DataFrame()
    gps_activity = pd.DataFrame()

In [ ]:
# Load gyroscope data
try:
    gyro_df = load_subject_data(DATA_DIR, SUBJECT_ID, 'gyro')
    print(f"Loaded {len(gyro_df)} gyroscope samples")
    print(f"Gyro date range: {gyro_df['timestamp'].min()} to {gyro_df['timestamp'].max()}")
    gyro_activity = compute_gyro_activity(gyro_df)
except FileNotFoundError:
    print("No gyroscope data found for this subject")
    gyro_df = pd.DataFrame()
    gyro_activity = pd.DataFrame()

In [ ]:
# Merge all sensor data
merged_df = merge_sensor_data(wear_df, gps_activity, gyro_activity)

print(f"Merged data: {len(merged_df)} minutes")
print(f"  With GPS: {merged_df['has_gps'].sum()}")
print(f"  With Gyro: {merged_df['has_gyro'].sum()}")

merged_df.head(10)

In [ ]:
# Compute concordance metrics
concordance = compute_concordance_metrics(merged_df)

print("Cross-Sensor Concordance Metrics:")
print("=" * 50)
for key, value in concordance.items():
    if isinstance(value, float) and not np.isnan(value):
        print(f"{key}: {value:.3f}")
    else:
        print(f"{key}: {value}")

In [ ]:
# Visualize concordance: Accelerometer magnitude vs GPS displacement
if merged_df['has_gps'].sum() > 0:
    gps_merged = merged_df[merged_df['has_gps'] & merged_df['valid']].copy()
    
    fig, ax = plt.subplots(figsize=(10, 6))
    
    colors = gps_merged['wear'].map({True: 'green', False: 'red'})
    ax.scatter(gps_merged['mean_magnitude'], gps_merged['displacement_m'],
               c=colors, alpha=0.5, s=20)
    
    ax.set_xlabel('Accelerometer Mean Magnitude (g)')
    ax.set_ylabel('GPS Displacement (m)')
    ax.set_title('Accelerometer vs GPS Activity\n(Green=Wear, Red=Non-wear)')
    ax.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
else:
    print("No GPS data available for visualization.")

In [ ]:
# Visualize concordance: Accelerometer magnitude vs Gyro activity
if merged_df['has_gyro'].sum() > 0:
    gyro_merged = merged_df[merged_df['has_gyro'] & merged_df['valid']].copy()
    
    fig, ax = plt.subplots(figsize=(10, 6))
    
    colors = gyro_merged['wear'].map({True: 'green', False: 'red'})
    ax.scatter(gyro_merged['mean_magnitude'], gyro_merged['mean_angular_rate'],
               c=colors, alpha=0.5, s=20)
    
    ax.set_xlabel('Accelerometer Mean Magnitude (g)')
    ax.set_ylabel('Gyroscope Mean Angular Rate')
    ax.set_title('Accelerometer vs Gyroscope Activity\n(Green=Wear, Red=Non-wear)')
    ax.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
else:
    print("No gyroscope data available for visualization.")

---

## Step 7: Sensitivity Analysis

Explore how the results change with different parameter choices.

In [ ]:
# Sensitivity to alpha (significance level)
alphas = [0.001, 0.01, 0.05, 0.10, 0.20]
results = []

for alpha in alphas:
    test_params = WearDetectionParams(
        noise_sigma=noise_sigma,
        alpha=alpha,
        min_samples_per_minute=MIN_SAMPLES_PER_MINUTE
    )
    test_wear = detect_wear_minutes(acc_df, test_params)
    valid = test_wear['valid'].sum()
    wear = test_wear['wear'].sum()
    results.append({
        'alpha': alpha,
        'critical_value': get_critical_value(alpha),
        'wear_minutes': wear,
        'wear_fraction': wear / valid if valid > 0 else 0
    })

sensitivity_df = pd.DataFrame(results)
print("Sensitivity to α:")
sensitivity_df

In [ ]:
# Visualize sensitivity
fig, ax = plt.subplots(figsize=(10, 5))

ax.plot(sensitivity_df['alpha'], sensitivity_df['wear_fraction'] * 100, 
        'o-', markersize=10, linewidth=2)

ax.set_xlabel('Significance Level (α)')
ax.set_ylabel('Wear Fraction (%)')
ax.set_title('Sensitivity of Wear Classification to α')
ax.grid(True, alpha=0.3)

# Annotate
for _, row in sensitivity_df.iterrows():
    ax.annotate(f"{row['wear_fraction']*100:.1f}%", 
                (row['alpha'], row['wear_fraction']*100),
                textcoords="offset points", xytext=(0,10), ha='center')

plt.tight_layout()
plt.show()

In [ ]:
# Sensitivity to noise sigma
sigmas = [noise_sigma * 0.5, noise_sigma, noise_sigma * 2, noise_sigma * 5]
sigma_results = []

for sigma in sigmas:
    test_params = WearDetectionParams(
        noise_sigma=sigma,
        alpha=ALPHA,
        min_samples_per_minute=MIN_SAMPLES_PER_MINUTE
    )
    test_wear = detect_wear_minutes(acc_df, test_params)
    valid = test_wear['valid'].sum()
    wear = test_wear['wear'].sum()
    sigma_results.append({
        'sigma': sigma,
        'sigma_mg': sigma * 1000,
        'wear_minutes': wear,
        'wear_fraction': wear / valid if valid > 0 else 0
    })

sigma_sensitivity = pd.DataFrame(sigma_results)
print("Sensitivity to noise σ:")
sigma_sensitivity

---

## Summary & Next Steps

### Key Findings for This Subject

In [ ]:
print(f"="*60)
print(f"SUMMARY FOR SUBJECT: {SUBJECT_ID}")
print(f"="*60)
print(f"")
print(f"Data Overview:")
print(f"  Total accelerometer samples: {len(acc_df):,}")
print(f"  Date range: {acc_df['timestamp'].min().date()} to {acc_df['timestamp'].max().date()}")
print(f"")
print(f"Noise Estimation:")
print(f"  Estimated σ: {noise_sigma:.4f} g ({noise_sigma*1000:.2f} mg)")
print(f"")
print(f"Wear Detection (α={ALPHA}):")
print(f"  Total minutes classified: {len(wear_df)}")
print(f"  Valid minutes: {wear_df['valid'].sum()}")
print(f"  Wear minutes: {wear_df['wear'].sum()}")
print(f"  Overall wear fraction: {wear_df['wear'].sum() / wear_df['valid'].sum() * 100:.1f}%")
print(f"")
print(f"Cross-Sensor Concordance:")
print(f"  GPS coverage: {concordance['gps_coverage']*100:.1f}%" if concordance['gps_coverage'] else "  GPS: No data")
print(f"  Gyro coverage: {concordance['gyro_coverage']*100:.1f}%" if concordance['gyro_coverage'] else "  Gyro: No data")

### Next Steps

1. **Run on all subjects**: Use the full pipeline to process all 6 participants
2. **Compare daily patterns**: Look for consistency/variability across subjects
3. **Validate with GPS**: Check if accelerometer "wear" aligns with GPS movement
4. **Decide on non-wear handling**: Choose imputation strategy based on observed patterns
5. **Compute step counts**: Apply step detection to wear periods only